# CKD — ANOVA + DODA (Rank Fusion)

Real, raw UCI Chronic Kidney Disease data (400 patients, Rubini/Soundarapandian/Eswaran 2015) — **not** the pre-cleaned CSV, because imputation needs to happen *after* the train/test split (fit on train only) to avoid leaking test-set information into the imputed values. This notebook does that split-then-impute correctly, matching the leakage-awareness principle already established elsewhere in this project.

**Two things about this dataset that the breast cancer notebooks didn't need to handle:**
1. **~10% missing values overall** (up to 38% for `rbc`) — handled with median imputation for lab values and mode imputation for categorical findings, both fit on the training set only.
2. **Mild class imbalance** (62.5% CKD / 37.5% not-CKD) — similar magnitude to breast cancer's ~63/37 split, so `class_weight="balanced"` (already the convention throughout this project) is used rather than resampling (SMOTE etc.), which would be overkill for this level of imbalance.

**Fusion mechanism:** this notebook uses **Rank Fusion** instead of the Hadamard (multiplicative) fusion used in the breast cancer notebooks.

In [2]:
# =============================================================================
# STEP 1: LOAD RAW DATASET
# =============================================================================

import pandas as pd
import numpy as np

df = pd.read_csv(
    "../../data/raw/ckd.csv"
)

df = df.drop(columns=["id"])

print("=" * 70)
print("RAW DATASET")
print("=" * 70)

print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

display(df.head())

RAW DATASET
Rows    : 400
Columns : 25


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [3]:
# =============================================================================
# STEP 2: CLEAN KNOWN DATA-QUALITY ISSUES (before anything else)
# =============================================================================

# This exact UCI file has two well-documented quirks (see 01_ckd_eda.ipynb):
#   1. Stray whitespace/tabs turn some numeric columns (pcv, wc, rc) into text,
#      and cause a "ckd\t" vs "ckd" inconsistency in the label column.
#   2. Categorical columns use text values (yes/no, normal/abnormal, etc.)
#      that need numeric encoding before imputation and modeling.

print("=" * 70)
print("BEFORE CLEANUP")
print("=" * 70)
print("classification unique:", df["classification"].unique().tolist())
print("pcv dtype:", df["pcv"].dtype, "| wc dtype:", df["wc"].dtype, "| rc dtype:", df["rc"].dtype)

# Strip stray whitespace/tabs from every text column
categorical_cols_raw = df.select_dtypes(include="object").columns
for col in categorical_cols_raw:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"nan": np.nan, "?": np.nan})

# pcv / wc / rc are genuinely numeric lab values
for col in ["pcv", "wc", "rc"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\n" + "=" * 70)
print("AFTER CLEANUP")
print("=" * 70)
print("classification unique:", df["classification"].unique().tolist())
print("pcv dtype:", df["pcv"].dtype, "| wc dtype:", df["wc"].dtype, "| rc dtype:", df["rc"].dtype)

BEFORE CLEANUP
classification unique: ['ckd', 'ckd\t', 'notckd']
pcv dtype: str | wc dtype: str | rc dtype: str

AFTER CLEANUP
classification unique: ['ckd', 'notckd']
pcv dtype: float64 | wc dtype: float64 | rc dtype: float64


C:\Users\johnm\AppData\Local\Temp\ipykernel_24788\1651612034.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols_raw = df.select_dtypes(include="object").columns


In [4]:
# =============================================================================
# STEP 3: ENCODE CATEGORICAL FEATURES AND TARGET
# =============================================================================

target_column = "target"

# Binary/nominal categorical features -> 0/1 (missing values stay NaN for now,
# handled explicitly by imputation in Step 6 rather than silently by the mapping)
binary_maps = {
    "rbc":   {"normal": 0, "abnormal": 1},
    "pc":    {"normal": 0, "abnormal": 1},
    "pcc":   {"notpresent": 0, "present": 1},
    "ba":    {"notpresent": 0, "present": 1},
    "htn":   {"no": 0, "yes": 1},
    "dm":    {"no": 0, "yes": 1},
    "cad":   {"no": 0, "yes": 1},
    "appet": {"poor": 0, "good": 1},
    "pe":    {"no": 0, "yes": 1},
    "ane":   {"no": 0, "yes": 1},
}

for col, mapping in binary_maps.items():
    df[col] = df[col].map(mapping)

df[target_column] = df["classification"].map({"ckd": 1, "notckd": 0})
df = df.drop(columns=["classification"])

print("=" * 70)
print("ENCODED DATASET")
print("=" * 70)
display(df.head())

print("\n" + "=" * 70)
print("TARGET DISTRIBUTION (0 = not CKD, 1 = CKD)")
print("=" * 70)
display(df[target_column].value_counts())
display((df[target_column].value_counts(normalize=True) * 100).round(2))
print("\nMild imbalance (~63/37) — comparable to the breast cancer dataset\'s")
print("~63/37 split. class_weight=\'balanced\' is used in the models below,")
print("consistent with how breast cancer was handled, rather than resampling.")

ENCODED DATASET


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,target
0,48.0,80.0,1.020,1.0,0.0,NaN,0.0,0.0,0.0,121.0,...,44.0,7800.0,5.2,1.0,1.0,0.0,1.0,0.0,0.0,1
1,7.0,50.0,1.020,4.0,0.0,NaN,0.0,0.0,0.0,NaN,...,38.0,6000.0,NaN,0.0,0.0,0.0,1.0,0.0,0.0,1
2,62.0,80.0,1.010,2.0,3.0,0.0,0.0,0.0,0.0,423.0,...,31.0,7500.0,NaN,0.0,1.0,0.0,0.0,0.0,1.0,1
3,48.0,70.0,1.005,4.0,0.0,0.0,1.0,1.0,0.0,117.0,...,32.0,6700.0,3.9,1.0,0.0,0.0,0.0,1.0,1.0,1
4,51.0,80.0,1.010,2.0,0.0,0.0,0.0,0.0,0.0,106.0,...,35.0,7300.0,4.6,0.0,0.0,0.0,1.0,0.0,0.0,1



TARGET DISTRIBUTION (0 = not CKD, 1 = CKD)


target
1    250
0    150
Name: count, dtype: int64

target
1    62.5
0    37.5
Name: proportion, dtype: float64


Mild imbalance (~63/37) — comparable to the breast cancer dataset's
~63/37 split. class_weight='balanced' is used in the models below,
consistent with how breast cancer was handled, rather than resampling.


In [5]:
# =============================================================================
# STEP 4: FEATURE AND TARGET SEPARATION
# =============================================================================

X = df.drop(target_column, axis=1)
y = df[target_column]

print("=" * 70)
print("FEATURE MATRIX (X)")
print("=" * 70)
print(f"Rows    : {X.shape[0]}")
print(f"Columns : {X.shape[1]}")

print("\n" + "=" * 70)
print("MISSING VALUES BEFORE IMPUTATION")
print("=" * 70)
missing_summary = pd.DataFrame({
    "Missing Count": X.isnull().sum(),
    "Missing %": (X.isnull().sum() / len(X) * 100).round(2)
}).sort_values("Missing %", ascending=False)
display(missing_summary[missing_summary["Missing Count"] > 0])

total_missing = X.isnull().sum().sum()
print(f"\nTotal missing: {total_missing} / {X.shape[0]*X.shape[1]} "
      f"({total_missing/(X.shape[0]*X.shape[1])*100:.2f}%)")

FEATURE MATRIX (X)
Rows    : 400
Columns : 24

MISSING VALUES BEFORE IMPUTATION


,Missing Count,Missing %
rbc,152,38.00
rc,131,32.75
wc,106,26.50
pot,88,22.00
sod,87,21.75
pcv,71,17.75
pc,65,16.25
hemo,52,13.00
su,49,12.25
sg,47,11.75



Total missing: 1012 / 9600 (10.54%)


In [6]:
# =============================================================================
# STEP 5: TRAIN-TEST SPLIT (BEFORE imputation and scaling — avoids leakage)
# =============================================================================

from sklearn.model_selection import train_test_split

print("=" * 70)
print("TRAIN-TEST SPLIT")
print("=" * 70)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")

print("\nTraining target distribution:")
display((y_train.value_counts(normalize=True) * 100).round(2))
print("Testing target distribution:")
display((y_test.value_counts(normalize=True) * 100).round(2))

TRAIN-TEST SPLIT
X_train shape : (320, 24)
X_test shape  : (80, 24)

Training target distribution:


target
1    62.5
0    37.5
Name: proportion, dtype: float64

Testing target distribution:


target
1    62.5
0    37.5
Name: proportion, dtype: float64

In [7]:
# =============================================================================
# STEP 6: IMPUTATION — fit on TRAINING data only, apply to both
# =============================================================================

from sklearn.impute import SimpleImputer

# Numerical lab values: median imputation. Median (not mean) because several
# of these (sc, bu, su) are right-skewed with real clinical outliers — a mean
# imputer would be pulled toward those extreme values.
numerical_features = [
    "age", "bp", "sg", "al", "su", "bgr", "bu", "sc",
    "sod", "pot", "hemo", "pcv", "wc", "rc"
]

# Categorical/binary clinical findings: most-frequent (mode) imputation —
# standard practice for nominal/binary features where a median is meaningless.
categorical_features = [
    "rbc", "pc", "pcc", "ba", "htn", "dm", "cad", "appet", "pe", "ane"
]

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

# Fit on TRAIN only
num_imputer.fit(X_train[numerical_features])
cat_imputer.fit(X_train[categorical_features])

X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()

X_train_imputed[numerical_features] = num_imputer.transform(X_train[numerical_features])
X_test_imputed[numerical_features] = num_imputer.transform(X_test[numerical_features])

X_train_imputed[categorical_features] = cat_imputer.transform(X_train[categorical_features])
X_test_imputed[categorical_features] = cat_imputer.transform(X_test[categorical_features])

print("=" * 70)
print("IMPUTATION COMPLETE")
print("=" * 70)
print("Missing values remaining in X_train:", X_train_imputed.isnull().sum().sum())
print("Missing values remaining in X_test :", X_test_imputed.isnull().sum().sum())

print("\nMedian values learned from TRAINING data (used to fill both train and test):")
display(pd.Series(num_imputer.statistics_, index=numerical_features))

print("\nMode values learned from TRAINING data:")
display(pd.Series(cat_imputer.statistics_, index=categorical_features))

IMPUTATION COMPLETE
Missing values remaining in X_train: 0
Missing values remaining in X_test : 0

Median values learned from TRAINING data (used to fill both train and test):


age       54.00
bp        80.00
sg         1.02
al         0.00
su         0.00
bgr      120.50
bu        42.00
sc         1.20
sod      138.00
pot        4.30
hemo      12.60
pcv       40.00
wc      8100.00
rc         4.80
dtype: float64


Mode values learned from TRAINING data:


rbc      0.0
pc       0.0
pcc      0.0
ba       0.0
htn      0.0
dm       0.0
cad      0.0
appet    1.0
pe       0.0
ane      0.0
dtype: float64

In [8]:
# =============================================================================
# STEP 7: FEATURE SCALING (fit on TRAINING data only)
# =============================================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_imputed.columns).reset_index(drop=True)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_imputed.columns).reset_index(drop=True)

y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("=" * 70)
print("SCALING COMPLETE")
print("=" * 70)
print(f"X_train_scaled shape : {X_train_scaled.shape}")
print(f"X_test_scaled shape  : {X_test_scaled.shape}")
display(X_train_scaled.head())

SCALING COMPLETE
X_train_scaled shape : (320, 24)
X_test_scaled shape  : (80, 24)


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane
0,-0.364027,0.224371,-0.508859,-0.678342,-0.347672,-0.356034,2.019744,-0.372545,-0.258199,-0.491252,...,-0.264295,-0.235403,0.708759,-0.650709,-0.75918,-0.684025,-0.291111,0.490214,-0.470504,-0.425220
1,0.463391,1.660345,-0.508859,2.411616,1.712604,-0.356034,-0.495112,-0.372545,-0.258199,1.577914,...,-1.894210,-2.324464,0.557632,-1.009720,1.31721,1.461935,3.435113,0.490214,-0.470504,2.351725
2,-0.364027,-1.211603,-1.444691,0.866637,-0.347672,-0.356034,2.019744,2.684237,-0.258199,1.759665,...,-0.916261,-1.218491,0.330943,0.067315,1.31721,1.461935,-0.291111,0.490214,-0.470504,-0.425220
3,1.054403,-1.211603,0.426974,-0.678342,-0.347672,-0.356034,-0.495112,-0.372545,-0.258199,0.403522,...,0.025467,0.133254,-0.084654,0.067315,1.31721,-0.684025,-0.291111,-2.039924,-0.470504,-0.425220
4,-0.186723,2.378332,-0.508859,1.639126,-0.347672,2.808717,-0.495112,2.684237,-0.258199,-0.505233,...,-1.423346,-1.587148,-1.255883,-2.685108,1.31721,-0.684025,3.435113,0.490214,-0.470504,2.351725


# ANOVA

## 1. Baseline

In [9]:
# =============================================================================
# ANOVA BASELINE FEATURE SELECTION (TOP-K = 5,10,15,20)
# =============================================================================

from sklearn.feature_selection import SelectKBest, f_classif
import pandas as pd

anova_selector = SelectKBest(score_func=f_classif, k="all")
anova_selector.fit(X_train_scaled, y_train)

anova_scores = pd.DataFrame({
    "Feature": X_train_scaled.columns,
    "ANOVA Score": anova_selector.scores_
}).sort_values(by="ANOVA Score", ascending=False).reset_index(drop=True)

k_values = [5, 10, 15, 20]
anova_results = {}

for k in k_values:
    top_features = anova_scores.head(k)["Feature"].tolist()
    mask = X_train_scaled.columns.isin(top_features)

    anova_results[k] = {
        "features": top_features,
        "X_train": X_train_scaled.loc[:, mask],
        "X_test": X_test_scaled.loc[:, mask],
        "scores": anova_scores.head(k)
    }

    print("=" * 70)
    print(f"TOP {k} ANOVA FEATURES")
    print("=" * 70)
    print(top_features)
    print("\nScores:")
    print(anova_scores.head(k))

TOP 5 ANOVA FEATURES
['hemo', 'pcv', 'sg', 'rc', 'htn']

Scores:
  Feature  ANOVA Score
0    hemo   379.745186
1     pcv   264.061532
2      sg   226.366547
3      rc   178.710132
4     htn   168.099398
TOP 10 ANOVA FEATURES
['hemo', 'pcv', 'sg', 'rc', 'htn', 'dm', 'al', 'pc', 'appet', 'bu']

Scores:
  Feature  ANOVA Score
0    hemo   379.745186
1     pcv   264.061532
2      sg   226.366547
3      rc   178.710132
4     htn   168.099398
5      dm   124.117347
6      al   121.280532
7      pc    54.837591
8   appet    53.576087
9      bu    53.326008
TOP 15 ANOVA FEATURES
['hemo', 'pcv', 'sg', 'rc', 'htn', 'dm', 'al', 'pc', 'appet', 'bu', 'pe', 'bgr', 'ane', 'sod', 'bp']

Scores:
   Feature  ANOVA Score
0     hemo   379.745186
1      pcv   264.061532
2       sg   226.366547
3       rc   178.710132
4      htn   168.099398
5       dm   124.117347
6       al   121.280532
7       pc    54.837591
8    appet    53.576087
9       bu    53.326008
10      pe    48.707746
11     bgr    47.414857
1

In [10]:
# =============================================================================
# BASELINE MODELS
# =============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=42
    ),
    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
    )
}

In [11]:
# =============================================================================
# MODEL EVALUATION
# =============================================================================

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

results = []

for k in k_values:
    Xtr = anova_results[k]["X_train"]
    Xts = anova_results[k]["X_test"]
    features = ", ".join(anova_results[k]["features"])

    for model_name, model in models.items():
        model.fit(Xtr, y_train)
        y_pred = model.predict(Xts)
        y_prob = model.predict_proba(Xts)[:, 1]

        results.append({
            "Method": "ANOVA",
            "Top-K": k,
            "Model": model_name,
            "Selected Features": features,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, y_prob)
        })

results_df = pd.DataFrame(results)
display(results_df)

,Method,Top-K,Model,Selected Features,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANOVA,5,Logistic Regression,"hemo, pcv, sg, rc, htn",0.9500,0.979167,0.94,0.959184,0.986000
1,ANOVA,5,Random Forest,"hemo, pcv, sg, rc, htn",0.9875,0.980392,1.00,0.990099,0.998667
2,ANOVA,5,XGBoost,"hemo, pcv, sg, rc, htn",0.9875,0.980392,1.00,0.990099,0.998333
3,ANOVA,10,Logistic Regression,"hemo, pcv, sg, rc, htn, dm, al, pc, appet, bu",0.9750,1.000000,0.96,0.979592,1.000000
4,ANOVA,10,Random Forest,"hemo, pcv, sg, rc, htn, dm, al, pc, appet, bu",0.9875,1.000000,0.98,0.989899,1.000000
5,ANOVA,10,XGBoost,"hemo, pcv, sg, rc, htn, dm, al, pc, appet, bu",1.0000,1.000000,1.00,1.000000,1.000000
6,ANOVA,15,Logistic Regression,"hemo, pcv, sg, rc, htn, dm, al, pc, appet, bu,...",0.9750,1.000000,0.96,0.979592,1.000000
7,ANOVA,15,Random Forest,"hemo, pcv, sg, rc, htn, dm, al, pc, appet, bu,...",1.0000,1.000000,1.00,1.000000,1.000000
8,ANOVA,15,XGBoost,"hemo, pcv, sg, rc, htn, dm, al, pc, appet, bu,...",0.9875,1.000000,0.98,0.989899,1.000000
9,ANOVA,20,Logistic Regression,"hemo, pcv, sg, rc, htn, dm, al, pc, appet, bu,...",0.9750,1.000000,0.96,0.979592,0.999333


In [16]:
results_df.to_csv(
    "../../../results/ckd/baseline_results/anova_baseline_results.csv",
    index=False
)

## 2. DODA (Rank Fusion)

In [17]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m", "pip", "install", "--upgrade", "--force-reinstall", "--no-cache-dir",
    "git+https://github.com/anandha-3679/DODA.git"
])

0

In [18]:
import sys
import doda
from doda import DODASelector
import inspect

print("Python:", sys.executable)
print("DODA:", doda.__file__)
print("Selector:", inspect.getfile(DODASelector))

Python: c:\Users\johnm\msc_research\.venv\Scripts\python.exe
DODA: c:\Users\johnm\msc_research\.venv\Lib\site-packages\doda\__init__.py
Selector: c:\Users\johnm\msc_research\.venv\Lib\site-packages\doda\selector.py


### Define RankFusion

**It is defined locally here so this notebook can actually run. It implements the same interface as `HadamardFusion` (`BaseFusion.fuse(math_scores, clinical_weights)`), so `DODASelector` uses it exactly the same way.

**What it does, mechanically:** instead of multiplying a feature's normalized statistical score by its clinical weight (Hadamard fusion), it converts *both* the statistical score and the clinical weight into rank positions (1 = best), sums the two ranks, and inverts the sum into a score where higher = better — a standard Borda-count-style rank fusion. This sidesteps the exact dynamic-range problem flagged earlier in this project (raw statistical scores spanning hundreds of times the range of clinical weights): once everything is a rank position, a feature ranked 1st clinically and 20th statistically is treated identically regardless of how large the underlying statistical score actually was.

In [19]:
from doda.fusion.base import BaseFusion


class RankFusion(BaseFusion):
    """
    Rank-based fusion: combines the statistical (mathematical) ranking and
    the clinical-weight ranking by rank position rather than by raw score
    magnitude.
    """

    def fuse(self, math_scores, clinical_weights):

        # -----------------------------------------------------------------
        # Rank features by normalized mathematical score (1 = highest score)
        # -----------------------------------------------------------------
        math_ranked = sorted(
            math_scores.items(), key=lambda x: x[1], reverse=True
        )
        math_rank = {
            feature: idx + 1
            for idx, (feature, _) in enumerate(math_ranked)
        }

        # -----------------------------------------------------------------
        # Rank features by clinical weight (1 = highest weight)
        # -----------------------------------------------------------------
        clinical_ranked = sorted(
            clinical_weights.items(), key=lambda x: x[1], reverse=True
        )
        clinical_rank = {
            feature: idx + 1
            for idx, (feature, _) in enumerate(clinical_ranked)
        }

        # -----------------------------------------------------------------
        # Combine ranks (lower combined rank = more important), then invert
        # into a score where HIGHER = more important, matching the
        # convention DODASelector / TopKRanker expect (same convention
        # HadamardFusion's output already follows).
        # -----------------------------------------------------------------
        n = len(math_scores)
        final_scores = {}

        for feature in math_scores:
            combined_rank_sum = (
                math_rank[feature]
                + clinical_rank.get(feature, n)
            )
            final_scores[feature] = (2 * n) - combined_rank_sum

        return final_scores

In [21]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

from doda import DODASelector

from doda.adapters import SklearnAdapter

from doda.knowledge.providers import JSONProvider

# NOTE: RankFusion imported from the LOCAL definition above, not from
# doda.fusion (it doesn\'t exist in the package yet — see markdown note).

In [23]:
# =============================================================================
# ANOVA + RANK FUSION DODA FEATURE SELECTION
# =============================================================================

import pandas as pd

k_values = [5, 10, 15, 20]

provider = JSONProvider(
    "../../../config/clinical_weights/ckd_clinical_weights.json"
)

# Rank-based fusion instead of Hadamard multiplication
fusion = RankFusion()

rank_doda_results = {}

for k in k_values:

    print("\n" + "=" * 70)
    print(f"ANOVA + RANK FUSION DODA \u2014 TOP {k} FEATURES")
    print("=" * 70)

    anova = SklearnAdapter(
        SelectKBest(score_func=f_classif, k=k)
    )

    selector = DODASelector(
        operators=[anova],
        provider=provider,
        fusion=fusion,
        top_k=k
    )

    X_train_selected = selector.fit_transform(X_train_scaled, y_train)
    X_test_selected = selector.transform(X_test_scaled)

    selected_features = selector.get_selected_features()

    print("\nSelected Features:")
    print(selected_features)

    rank_doda_results[k] = {
        "selector": selector,
        "X_train": X_train_selected,
        "X_test": X_test_selected,
        "features": selected_features,
        "raw_math_scores": selector.raw_math_scores_,
        "math_scores": selector.math_scores_,
        "clinical_weights": selector.clinical_weights_,
        "final_scores": selector.final_scores_
    }

print("\n" + "=" * 70)
print("ANOVA + RANK FUSION DODA COMPLETED")
print("=" * 70)


ANOVA + RANK FUSION DODA — TOP 5 FEATURES
Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x00000212C84560D0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(13.033768241197253), 'bp': np.float64(33.37626382493944), 'sg': np.float64(226.36654745916326), 'al': np.float64(121.28053233555288), 'su': np.float64(24.866490274599517), 'rbc': np.float64(26.176829268292586), 'pc': np.float64(54.83759124087606), 'pcc': np.float64(28.88664596273283), 'ba': np.float64(13.249999999999986), 'bgr': np.float64(47.41485748565791), 'bu': np.float64(53.326007627262655), 'sc': np.float64(27.84285915023033), 'sod': np.float64(37.26240983413097), 'pot': np.float64(0.11007057871478018), 'hemo': np.float64(379.74518553534807), 'pcv': np.float64(264.0615317634784), 'wc': np.float64(9.43

## Cell 2 — Inspect Mathematical Scores

This remains exactly the same, because Rank Fusion does not change how the mathematical selector produces its scores.

In [24]:
k = 10

selector = rank_doda_results[k]["selector"]

raw_math_scores = pd.DataFrame(
    selector.raw_math_scores_.items(),
    columns=["Feature", "Raw Math Score"]
)

normalized_math_scores = pd.DataFrame(
    selector.math_scores_.items(),
    columns=["Feature", "Normalized Math Score"]
)

math_scores = raw_math_scores.merge(normalized_math_scores, on="Feature")
math_scores = math_scores.sort_values(by="Normalized Math Score", ascending=False)

print("=" * 70)
print("MATHEMATICAL SCORES")
print("=" * 70)
display(math_scores)

MATHEMATICAL SCORES


,Feature,Raw Math Score,Normalized Math Score
14,hemo,379.745186,1.000000
15,pcv,264.061532,0.695365
2,sg,226.366547,0.596101
17,rc,178.710132,0.470605
18,htn,168.099398,0.442664
19,dm,124.117347,0.326844
3,al,121.280532,0.319373
6,pc,54.837591,0.144406
21,appet,53.576087,0.141084
10,bu,53.326008,0.140426


## Cell 3 — Inspect Clinical Weights

Again, this stays the same.

In [25]:
clinical_weights = pd.DataFrame(
    selector.clinical_weights_.items(),
    columns=["Feature", "Clinical Weight"]
)

clinical_weights = clinical_weights.sort_values(by="Clinical Weight", ascending=False)

print("=" * 70)
print("CLINICAL WEIGHTS")
print("=" * 70)
display(clinical_weights)

CLINICAL WEIGHTS


,Feature,Clinical Weight
3,al,1.0
11,sc,1.0
19,dm,0.8
5,rbc,0.8
18,htn,0.8
10,bu,0.7
1,bp,0.7
14,hemo,0.7
13,pot,0.7
23,ane,0.6


## Cell 4 — Inspect Final Rank Fusion Scores

Labelled **Final Rank Fusion Score** rather than *Final DODA Score*, since this is specifically testing the fusion mechanism.

In [26]:
final_scores = pd.DataFrame(
    selector.final_scores_.items(),
    columns=["Feature", "Final Rank Fusion Score"]
)

final_scores = final_scores.sort_values(by="Final Rank Fusion Score", ascending=False)

print("=" * 70)
print("FINAL RANK FUSION SCORES")
print("=" * 70)
display(final_scores)

FINAL RANK FUSION SCORES


,Feature,Final Rank Fusion Score
3,al,40
18,htn,39
14,hemo,38
19,dm,37
15,pcv,33
10,bu,31
11,sc,29
2,sg,28
5,rbc,27
1,bp,27


In [27]:
selector.get_selected_features()

['al', 'htn', 'hemo', 'dm', 'pcv', 'bu', 'sc', 'sg', 'bp', 'rbc']

In [28]:
# =============================================================================
# COMPARE RANKINGS
# =============================================================================

comparison = selector.compare_scores()

display(comparison)


MATHEMATICAL vs CLINICAL vs RANKFUSION

Fusion Method: RankFusion

Top Features After Fusion:
  Feature  Math Rank  Clinical Rank  Final Rank  Rank Change
0      al          7              1           1            6
1     htn          5              4           2            3
2    hemo          1              9           3           -2
3      dm          6              5           4            2
4     pcv          2             13           5           -3
5      bu         10              7           6            4
6      sc         17              2           7           10
7      sg          3             17           8           -5
8      bp         15              6           9            6
9     rbc         18              3          10            8


,Feature,Math Rank,Normalized Math Score,Clinical Rank,Clinical Weight,Final Rank,Final Score,Rank Change
0,al,7,0.3194,1,1.0,1,40,6
1,htn,5,0.4427,4,0.8,2,39,3
2,hemo,1,1.0000,9,0.7,3,38,-2
3,dm,6,0.3268,5,0.8,4,37,2
4,pcv,2,0.6954,13,0.6,5,33,-3
5,bu,10,0.1404,7,0.7,6,31,4
6,sc,17,0.0733,2,1.0,7,29,10
7,sg,3,0.5961,17,0.5,8,28,-5
8,bp,15,0.0879,6,0.7,9,27,6
9,rbc,18,0.0689,3,0.8,10,27,8


## Cell 7 — Model Evaluation

Same evaluation structure, but use `rank_doda_results`.

In [29]:
# =============================================================================
# RANK FUSION DODA MODEL EVALUATION
# =============================================================================

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

rank_doda_model_results = []

for k in k_values:

    Xtr = rank_doda_results[k]["X_train"]
    Xts = rank_doda_results[k]["X_test"]
    features = ", ".join(rank_doda_results[k]["features"])

    for model_name, model in models.items():

        model.fit(Xtr, y_train)
        y_pred = model.predict(Xts)
        y_prob = model.predict_proba(Xts)[:, 1]

        rank_doda_model_results.append({
            "Method": "ANOVA + Rank Fusion DODA",
            "Top-K": k,
            "Model": model_name,
            "Selected Features": features,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, y_prob)
        })

rank_doda_results_df = pd.DataFrame(rank_doda_model_results)
display(rank_doda_results_df)

,Method,Top-K,Model,Selected Features,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANOVA + Rank Fusion DODA,5,Logistic Regression,"al, htn, hemo, dm, pcv",0.9375,0.978723,0.92,0.948454,0.979000
1,ANOVA + Rank Fusion DODA,5,Random Forest,"al, htn, hemo, dm, pcv",0.9625,0.979592,0.96,0.969697,0.984667
2,ANOVA + Rank Fusion DODA,5,XGBoost,"al, htn, hemo, dm, pcv",0.9500,0.979167,0.94,0.959184,0.985667
3,ANOVA + Rank Fusion DODA,10,Logistic Regression,"al, htn, hemo, dm, pcv, bu, sc, sg, bp, rbc",0.9750,1.000000,0.96,0.979592,0.998667
4,ANOVA + Rank Fusion DODA,10,Random Forest,"al, htn, hemo, dm, pcv, bu, sc, sg, bp, rbc",0.9875,1.000000,0.98,0.989899,1.000000
5,ANOVA + Rank Fusion DODA,10,XGBoost,"al, htn, hemo, dm, pcv, bu, sc, sg, bp, rbc",0.9875,0.980392,1.00,0.990099,1.000000
6,ANOVA + Rank Fusion DODA,15,Logistic Regression,"al, htn, hemo, dm, pcv, bu, sc, sg, bp, rbc, r...",0.9875,1.000000,0.98,0.989899,1.000000
7,ANOVA + Rank Fusion DODA,15,Random Forest,"al, htn, hemo, dm, pcv, bu, sc, sg, bp, rbc, r...",1.0000,1.000000,1.00,1.000000,1.000000
8,ANOVA + Rank Fusion DODA,15,XGBoost,"al, htn, hemo, dm, pcv, bu, sc, sg, bp, rbc, r...",1.0000,1.000000,1.00,1.000000,1.000000
9,ANOVA + Rank Fusion DODA,20,Logistic Regression,"al, htn, hemo, dm, pcv, bu, sc, sg, bp, rbc, r...",0.9750,1.000000,0.96,0.979592,1.000000


In [30]:
rank_doda_results_df.to_csv(
    "../../../results/ckd/doda_results/anova_rankfusion_doda_results.csv",
    index=False
)

In [31]:
# =============================================================================
# COMBINED COMPARISON
# =============================================================================

comparison_df = pd.concat(
    [results_df, rank_doda_results_df],
    ignore_index=True
)

display(comparison_df)

,Method,Top-K,Model,Selected Features,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANOVA,5,Logistic Regression,"hemo, pcv, sg, rc, htn",0.9500,0.979167,0.94,0.959184,0.986000
1,ANOVA,5,Random Forest,"hemo, pcv, sg, rc, htn",0.9875,0.980392,1.00,0.990099,0.998667
2,ANOVA,5,XGBoost,"hemo, pcv, sg, rc, htn",0.9875,0.980392,1.00,0.990099,0.998333
3,ANOVA,10,Logistic Regression,"hemo, pcv, sg, rc, htn, dm, al, pc, appet, bu",0.9750,1.000000,0.96,0.979592,1.000000
4,ANOVA,10,Random Forest,"hemo, pcv, sg, rc, htn, dm, al, pc, appet, bu",0.9875,1.000000,0.98,0.989899,1.000000
5,ANOVA,10,XGBoost,"hemo, pcv, sg, rc, htn, dm, al, pc, appet, bu",1.0000,1.000000,1.00,1.000000,1.000000
6,ANOVA,15,Logistic Regression,"hemo, pcv, sg, rc, htn, dm, al, pc, appet, bu,...",0.9750,1.000000,0.96,0.979592,1.000000
7,ANOVA,15,Random Forest,"hemo, pcv, sg, rc, htn, dm, al, pc, appet, bu,...",1.0000,1.000000,1.00,1.000000,1.000000
8,ANOVA,15,XGBoost,"hemo, pcv, sg, rc, htn, dm, al, pc, appet, bu,...",0.9875,1.000000,0.98,0.989899,1.000000
9,ANOVA,20,Logistic Regression,"hemo, pcv, sg, rc, htn, dm, al, pc, appet, bu,...",0.9750,1.000000,0.96,0.979592,0.999333


In [32]:
for k in k_values:

    selector = rank_doda_results[k]["selector"]

    raw_rank = [
        x[0] for x in sorted(
            selector.raw_math_scores_.items(), key=lambda x: x[1], reverse=True
        )
    ]

    doda_rank = [
        x[0] for x in sorted(
            selector.final_scores_.items(), key=lambda x: x[1], reverse=True
        )
    ]

    print("\n" + "=" * 70)
    print(f"TOP {k}")
    print("=" * 70)

    print("Raw Math Ranking:")
    print(raw_rank)

    print("\nRank Fusion DODA Ranking:")
    print(doda_rank)

    print("\nTop-K Same:", raw_rank[:k] == doda_rank[:k])


TOP 5
Raw Math Ranking:
['hemo', 'pcv', 'sg', 'rc', 'htn', 'dm', 'al', 'pc', 'appet', 'bu', 'pe', 'bgr', 'ane', 'sod', 'bp', 'pcc', 'sc', 'rbc', 'su', 'cad', 'ba', 'age', 'wc', 'pot']

Rank Fusion DODA Ranking:
['al', 'htn', 'hemo', 'dm', 'pcv', 'bu', 'sc', 'sg', 'bp', 'rbc', 'rc', 'bgr', 'sod', 'pe', 'pc', 'ane', 'appet', 'age', 'pot', 'cad', 'pcc', 'su', 'ba', 'wc']

Top-K Same: False

TOP 10
Raw Math Ranking:
['hemo', 'pcv', 'sg', 'rc', 'htn', 'dm', 'al', 'pc', 'appet', 'bu', 'pe', 'bgr', 'ane', 'sod', 'bp', 'pcc', 'sc', 'rbc', 'su', 'cad', 'ba', 'age', 'wc', 'pot']

Rank Fusion DODA Ranking:
['al', 'htn', 'hemo', 'dm', 'pcv', 'bu', 'sc', 'sg', 'bp', 'rbc', 'rc', 'bgr', 'sod', 'pe', 'pc', 'ane', 'appet', 'age', 'pot', 'cad', 'pcc', 'su', 'ba', 'wc']

Top-K Same: False

TOP 15
Raw Math Ranking:
['hemo', 'pcv', 'sg', 'rc', 'htn', 'dm', 'al', 'pc', 'appet', 'bu', 'pe', 'bgr', 'ane', 'sod', 'bp', 'pcc', 'sc', 'rbc', 'su', 'cad', 'ba', 'age', 'wc', 'pot']

Rank Fusion DODA Ranking:
['al